# 1. Dataset Description

The **Facebook Live Sellers in Thailand** dataset contains 7,050 posts published by 10 Thai fashion and cosmetics sellers. This notebook downloads the data, validates its structure, stores an unchanged local copy, and performs an initial inspection before preprocessing.

## 1.1. Environment Setup

Synchronize the project environment with `uv sync` and select the `.venv` Python interpreter as the notebook kernel. The setup below locates the repository root so the notebook works whether it is launched from the project root or from the `notebooks` directory.

In [1]:
import logging
import sys
from pathlib import Path


def find_project_root(start_directory: Path) -> Path:
    """Return the nearest parent directory containing the source package."""
    candidates = (start_directory, *start_directory.parents)

    for candidate in candidates:
        if (candidate / "src").is_dir():
            return candidate

    raise RuntimeError("Could not locate the project root directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_DIRECTORY = PROJECT_ROOT / "src"
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "facebook_live_sellers.csv"

source_path = str(SOURCE_DIRECTORY)
if source_path not in sys.path:
    sys.path.insert(0, source_path)

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

LOGGER = logging.getLogger("facebook_clustering.data_collection")

### 1.1.1. Project Utilities

In [3]:
from data_collection_utils import (
    dataframe_overview,
    fetch_uci_features,
    save_dataframe_csv,
    validate_dataframe_contract,
)

## 1.2. Data Collection

The UCI Machine Learning Repository is the canonical source. Because this is an unsupervised learning project, the feature table is named `raw_posts`; no target variable is used.

In [4]:
UCI_DATASET_ID = 488

raw_posts = fetch_uci_features(UCI_DATASET_ID)

LOGGER.info("Dataset downloaded successfully: shape=%s", raw_posts.shape)

2026-07-19 20:08:12,046 | INFO | Dataset downloaded successfully: shape=(7050, 11)


### 1.2.1. Data Contract Validation

Validate the expected columns immediately after collection so that changes in the external data source fail early with a clear message.

In [5]:
REQUIRED_COLUMNS = (
    "status_type",
    "status_published",
    "num_reactions",
    "num_comments",
    "num_shares",
    "num_likes",
    "num_loves",
    "num_wows",
    "num_hahas",
    "num_sads",
    "num_angrys",
)

validate_dataframe_contract(
    raw_posts,
    required_columns=REQUIRED_COLUMNS,
)

LOGGER.info("Data contract validated successfully.")

2026-07-19 20:08:12,054 | INFO | Data contract validated successfully.


### 1.2.2. Raw Data Persistence

Save the collected feature table without transformations. This immutable snapshot separates data collection from the preprocessing performed in the next notebook.

In [6]:
raw_data_path = save_dataframe_csv(
    raw_posts,
    RAW_DATA_PATH,
)

LOGGER.info("Raw dataset saved to %s", raw_data_path)

2026-07-19 20:08:12,078 | INFO | Raw dataset saved to C:\Projetos\facebook_clusterizacao\data\raw\facebook_live_sellers.csv


## 1.3. Initial Data Inspection

Inspect the dataframe structure, completeness, cardinality, and a small sample before applying any transformation.

### 1.3.1. Dataset Metadata
| Characteristic | Detail |
| :--- | :--- |
| **Data Type** | Multivariate |
| **Subject Area** | Business / E-commerce |
| **ML Task** | Clustering |
| **Number of Instances** | 7,050 |
| **Attributes** | 11 |
| **Missing Values** | None (100% complete dataset) |

In [7]:
# Structural verification and data types
raw_posts.info()

display(dataframe_overview(raw_posts))

<class 'pandas.DataFrame'>
RangeIndex: 7050 entries, 0 to 7049
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   status_type       7050 non-null   str  
 1   status_published  7050 non-null   str  
 2   num_reactions     7050 non-null   int64
 3   num_comments      7050 non-null   int64
 4   num_shares        7050 non-null   int64
 5   num_likes         7050 non-null   int64
 6   num_loves         7050 non-null   int64
 7   num_wows          7050 non-null   int64
 8   num_hahas         7050 non-null   int64
 9   num_sads          7050 non-null   int64
 10  num_angrys        7050 non-null   int64
dtypes: int64(9), str(2)
memory usage: 606.0 KB


,dtype,non_null,missing,missing_pct,unique
column,,,,,
status_type,str,7050,0,0.0,4
status_published,str,7050,0,0.0,6913
num_reactions,int64,7050,0,0.0,1067
num_comments,int64,7050,0,0.0,993
num_shares,int64,7050,0,0.0,501
num_likes,int64,7050,0,0.0,1044
num_loves,int64,7050,0,0.0,229
num_wows,int64,7050,0,0.0,65
num_hahas,int64,7050,0,0.0,42


### 1.3.2. Sample Records

Display the first five observations to verify the original representation of each feature.

In [8]:
raw_posts.head()

,status_type,status_published,num_reactions,num_comments,num_shares,num_likes,num_loves,num_wows,num_hahas,num_sads,num_angrys
0,video,4/22/2018 6:00,529,512,262,432,92,3,1,1,0
1,photo,4/21/2018 22:45,150,0,0,150,0,0,0,0,0
2,video,4/21/2018 6:17,227,236,57,204,21,1,1,0,0
3,photo,4/21/2018 2:29,111,0,0,111,0,0,0,0,0
4,photo,4/18/2018 3:22,213,0,0,204,9,0,0,0,0


### 1.3.3. Data Dictionary

The table below documents the 11 attributes used throughout the project.

| # | Attribute | Data Type | Description |
| :-: | :--- | :---: | :--- |
| **0** | `status_type` | `str` (Nominal) | Type of publication on the platform (*video, photo, status, link*). |
| **1** | `status_published` | `str` (Temporal) | Exact publication date and time of the post. |
| **2** | `num_reactions` | `int64` | Total count of consolidated reactions on the post. |
| **3** | `num_comments` | `int64` | Total number of comments made on the publication. |
| **4** | `num_shares` | `int64` | Number of times the post was shared. |
| **5** | `num_likes` | `int64` | Specific count of "Like" reactions. |
| **6** | `num_loves` | `int64` | Specific count of "Love" reactions. |
| **7** | `num_wows` | `int64` | Specific count of "Wow" reactions. |
| **8** | `num_hahas` | `int64` | Specific count of "Haha" reactions. |
| **9** | `num_sads` | `int64` | Specific count of "Sad" reactions. |
| **10** | `num_angrys` | `int64` | Specific count of "Angry" reactions. |